# 04: Anomaly Detection, SHAP Explainability & Grounded GenAI
### Bridging Machine Learning with Explainable AI & Conversational Analytics

This notebook ties together the full end-to-end analytical architecture:
1. **Unsupervised Outlier Detection**: Isolation Forest + LOF + Category IQR bounds
2. **Explainable AI (XAI)**: SHAP TreeExplainer feature attributions
3. **Grounded GenAI**: Controlled natural language generation and conversational Q&A


In [ ]:
import os
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.anomaly_detection import detect_anomalies
from src.explainability import explain_next_month_forecast
from src.llm import FinancialAnalyst
print("Modules loaded.")


## 1. Spending Anomaly Radar


In [ ]:
anomalies_df, stats = detect_anomalies('../data/processed/cleaned_transactions.csv')
print(f"Total Transactions: {stats['total_transactions']}")
print(f"Detected Anomalies: {stats['anomalies_detected']} ({stats['anomaly_percentage']}%)")
display(anomalies_df.head(10))


## 2. SHAP Explainability: Why Did the Model Predict This Amount?


In [ ]:
explanation = explain_next_month_forecast()
print(f"Baseline Historical Spending: ?{explanation['base_value']:,.2f}")
print(f"Next Month Forecast:          ?{explanation['predicted_value']:,.2f}")
print(f"Net SHAP Adjustment:          ?{explanation['total_shap_adjustment']:+,.2f}")

impact_df = pd.DataFrame(explanation['feature_impacts'])
display(impact_df[['label', 'shap_impact_eur', 'value', 'direction']])


In [ ]:
plt.figure(figsize=(10, 5))
top_drivers = impact_df.head(7)
colors = ['#ef4444' if x > 0 else '#10b981' for x in top_drivers['shap_impact_eur']]
plt.barh(top_drivers['label'], top_drivers['shap_impact_eur'], color=colors)
plt.axvline(0, color='gray', linestyle='--')
plt.title("Top SHAP Feature Attributions on Next Month's Spending (?)", fontsize=13, fontweight='bold')
plt.xlabel("Impact on Predicted Spending (?)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Grounded GenAI & Conversational Analytics
The LLM does NOT calculate numbers or hallucinate predictions. It synthesizes verified ML facts.


In [ ]:
analyst = FinancialAnalyst()
print("--- Executive AI Briefing ---")
print(analyst.generate_executive_insights())


In [ ]:
queries = [
    "What were my 3 biggest purchases?",
    "How much did I spend on weekends compared with weekdays?",
    "Why is my predicted spending higher next month?",
    "Am I on track to stay under ?6,000?"
]

for q in queries:
    print(f"\nUSER: '{q}'")
    print(analyst.answer_query(q))
